## In This Notebook

This notebook covers the **first half** of the project pipeline — 
getting raw government data into a clean, queryable format.

By the end of this notebook you will have:

1. **Loaded** 500,000 rows from a 2.9GB real CMS Medicare CSV file
2. **Inspected** the data — understanding columns, data types, 
   and missing values
3. **Cleaned** the dataset — dropped unnecessary columns, 
   renamed cryptic CMS codes to human readable names
4. **Saved** everything into a SQLite database ready for SQL queries

This notebook does **not** do any analysis yet — that comes in the 
next notebook `02_sql_analytics.ipynb`. Think of this notebook as 
laying the foundation — garbage in, garbage out. Clean data first, 
insights second.

---
> **Key concept:** A data scientist spends 60-70% of their time 
> on exactly what this notebook does — loading, inspecting, and 
> cleaning data. The modeling and AI come later, but only work 
> well if this foundation is solid.

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
# Load 500,000 rows from the CMS Medicare dataset
FILE_PATH = 'data/MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv'
df = pd.read_csv(FILE_PATH,
                nrows = 500000,
                low_memory = False)
#Printing the result to make sure the sampling is successful
print(f"Length Of Rows:{len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"\nFirst look at the data:")
df.head(3)

Length Of Rows:500,000
Columns: 28

First look at the data:


,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_First_Name,Rndrng_Prvdr_MI,Rndrng_Prvdr_Crdntls,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_St1,Rndrng_Prvdr_St2,Rndrng_Prvdr_City,Rndrng_Prvdr_State_Abrvtn,...,HCPCS_Desc,HCPCS_Drug_Ind,Place_Of_Srvc,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Pymt_Amt,Avg_Mdcr_Stdzd_Amt
0,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Initial hospital care with straightforward or ...,N,F,12,12.0,12,250.226667,89.062500,60.312500,54.669167
1,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Initial hospital care with straightforward or ...,N,F,22,22.0,22,318.581818,130.312727,99.380000,98.429545
2,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Subsequent hospital care with straightforward ...,N,F,76,127.0,127,95.732283,54.820157,43.557323,38.748661


In [3]:
print("Column names and data types:")
print(df.info())
# Upon checking the sampling it seems like there are 6 columns which have null values and the result we found are not important and wont affect our analysis.

Column names and data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 28 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Rndrng_NPI                     500000 non-null  int64  
 1   Rndrng_Prvdr_Last_Org_Name     500000 non-null  object 
 2   Rndrng_Prvdr_First_Name        472228 non-null  object 
 3   Rndrng_Prvdr_MI                325704 non-null  object 
 4   Rndrng_Prvdr_Crdntls           443376 non-null  object 
 5   Rndrng_Prvdr_Ent_Cd            500000 non-null  object 
 6   Rndrng_Prvdr_St1               500000 non-null  object 
 7   Rndrng_Prvdr_St2               123181 non-null  object 
 8   Rndrng_Prvdr_City              500000 non-null  object 
 9   Rndrng_Prvdr_State_Abrvtn      500000 non-null  object 
 10  Rndrng_Prvdr_State_FIPS        500000 non-null  object 
 11  Rndrng_Prvdr_Zip5              500000 non-null  int64  
 12  R

In [12]:
missing= df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': missing_pct
})
print(missing_df[missing_df['missing_count'] > 0])

                         missing_count  missing_percent
Rndrng_Prvdr_First_Name          27772             5.55
Rndrng_Prvdr_MI                 174296            34.86
Rndrng_Prvdr_Crdntls             56624            11.32
Rndrng_Prvdr_St2                376819            75.36
Rndrng_Prvdr_RUCA                  409             0.08
Rndrng_Prvdr_RUCA_Desc             409             0.08


In [13]:
total_missing= df.isnull().sum().sum()
print(f"Total count of missing {total_missing:,}") # Total Missing values 636,329 out of 14,000,000 cells

Total count of missing 636,329


<h1>Cleaning & Renaming the Data</h1>
<b>Dropping columns we don't need</b> — middle initial, address line 2, RUCA codes (rural/urban classifications we won't use in this analysis).
<br><b>Rename columns to human readable names</b>

In [15]:
print(list(df.columns))

['Rndrng_NPI', 'Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_First_Name', 'Rndrng_Prvdr_MI', 'Rndrng_Prvdr_Crdntls', 'Rndrng_Prvdr_Ent_Cd', 'Rndrng_Prvdr_St1', 'Rndrng_Prvdr_St2', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_RUCA_Desc', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'HCPCS_Cd', 'HCPCS_Desc', 'HCPCS_Drug_Ind', 'Place_Of_Srvc', 'Tot_Benes', 'Tot_Srvcs', 'Tot_Bene_Day_Srvcs', 'Avg_Sbmtd_Chrg', 'Avg_Mdcr_Alowd_Amt', 'Avg_Mdcr_Pymt_Amt', 'Avg_Mdcr_Stdzd_Amt']


In [16]:
# Columns we don't need for billing analysis
columns_to_drop = [
    'Rndrng_Prvdr_MI',          # middle initial - not useful
    'Rndrng_Prvdr_St2',         # address line 2 - mostly empty
    'Rndrng_Prvdr_RUCA',        # rural/urban code - not needed
    'Rndrng_Prvdr_RUCA_Desc',   # rural/urban description - not needed
    'Rndrng_Prvdr_State_FIPS',  # FIPS code - we have state abbreviation
    'HCPCS_Drug_Ind',           # drug indicator - out of scope
    'Tot_Bene_Day_Srvcs',       # too granular for our analysis
]

df = df.drop(columns=columns_to_drop)

print(f"Columns before: 28")
print(f"Columns after:  {len(df.columns)}")
print(f"\nRemaining columns:")
print(list(df.columns))

Columns before: 28
Columns after:  21

Remaining columns:
['Rndrng_NPI', 'Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_First_Name', 'Rndrng_Prvdr_Crdntls', 'Rndrng_Prvdr_Ent_Cd', 'Rndrng_Prvdr_St1', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'HCPCS_Cd', 'HCPCS_Desc', 'Place_Of_Srvc', 'Tot_Benes', 'Tot_Srvcs', 'Avg_Sbmtd_Chrg', 'Avg_Mdcr_Alowd_Amt', 'Avg_Mdcr_Pymt_Amt', 'Avg_Mdcr_Stdzd_Amt']


In [17]:
# Rename columns to human readable names
df = df.rename(columns={
    'Rndrng_NPI'                  : 'provider_id',
    'Rndrng_Prvdr_Last_Org_Name'  : 'provider_name',
    'Rndrng_Prvdr_First_Name'     : 'first_name',
    'Rndrng_Prvdr_Crdntls'        : 'credentials',
    'Rndrng_Prvdr_Ent_Cd'         : 'entity_type',
    'Rndrng_Prvdr_St1'            : 'address',
    'Rndrng_Prvdr_City'           : 'city',
    'Rndrng_Prvdr_State_Abrvtn'   : 'state',
    'Rndrng_Prvdr_Zip5'           : 'zip_code',
    'Rndrng_Prvdr_Cntry'          : 'country',
    'Rndrng_Prvdr_Type'           : 'specialty',
    'Rndrng_Prvdr_Mdcr_Prtcptg_Ind' : 'medicare_participant',
    'HCPCS_Cd'                    : 'procedure_code',
    'HCPCS_Desc'                  : 'procedure_desc',
    'Place_Of_Srvc'               : 'place_of_service',
    'Tot_Benes'                   : 'total_patients',
    'Tot_Srvcs'                   : 'total_services',
    'Avg_Sbmtd_Chrg'              : 'avg_submitted_charge',
    'Avg_Mdcr_Alowd_Amt'          : 'avg_medicare_allowed',
    'Avg_Mdcr_Pymt_Amt'           : 'avg_medicare_payment',
    'Avg_Mdcr_Stdzd_Amt'          : 'avg_standardized_payment'
})

print("Columns renamed successfully")
print(list(df.columns))

Columns renamed successfully
['provider_id', 'provider_name', 'first_name', 'credentials', 'entity_type', 'address', 'city', 'state', 'zip_code', 'country', 'specialty', 'medicare_participant', 'procedure_code', 'procedure_desc', 'place_of_service', 'total_patients', 'total_services', 'avg_submitted_charge', 'avg_medicare_allowed', 'avg_medicare_payment', 'avg_standardized_payment']


In [24]:
import sqlite3
# Create a connection to SQLite database
# If the file doesn't exist, SQLite creates it automatically
conn = sqlite3.connect('data/cms_medicare.db')
# Save DataFrame to SQLite as a table called 'medicare_billing'
df.to_sql(
    name='medicare_billing',  # table name inside the database
    con=conn,                 # connection we just created
    if_exists='replace',      # if table exists, replace it
    index=False               # don't save the DataFrame index as a column
)

# Always close the connection when done
conn.close()

print("Data saved to SQLite successfully")
print("Database file: data/cms_medicare.db")
print(f"Table: medicare_billing")
print(f"Rows saved: {len(df):,}")

Data saved to SQLite successfully
Database file: data/cms_medicare.db
Table: medicare_billing
Rows saved: 500,000
